In [31]:
import pandas as pd


In [32]:
df = pd.read_csv("customer_churn_dataset-training-master.csv")
print(df.shape)
df.info()

(440833, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 440833 entries, 0 to 440832
Data columns (total 12 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   CustomerID         440832 non-null  float64
 1   Age                440832 non-null  float64
 2   Gender             440832 non-null  object 
 3   Tenure             440832 non-null  float64
 4   Usage Frequency    440832 non-null  float64
 5   Support Calls      440832 non-null  float64
 6   Payment Delay      440832 non-null  float64
 7   Subscription Type  440832 non-null  object 
 8   Contract Length    440832 non-null  object 
 9   Total Spend        440832 non-null  float64
 10  Last Interaction   440832 non-null  float64
 11  Churn              440832 non-null  float64
dtypes: float64(9), object(3)
memory usage: 40.4+ MB


In [33]:
# 2. column names (no spaces, lowercase)
df.columns = df.columns.str.lower().str.replace("customerid", "customer_id").str.replace(" ", "_")


In [34]:
# 3. missing values
print(df.isnull().sum())
df = df.dropna(how="all")          # one completely empty row


customer_id          1
age                  1
gender               1
tenure               1
usage_frequency      1
support_calls        1
payment_delay        1
subscription_type    1
contract_length      1
total_spend          1
last_interaction     1
churn                1
dtype: int64


In [35]:
# 4. duplicates
print("duplicates:", df.duplicated().sum())
print("duplicated ids:", df["customer_id"].duplicated().sum())
df = df.drop_duplicates(subset="customer_id")

duplicates: 0
duplicated ids: 0


In [36]:
print(df.isnull().sum())
df = df.dropna()              # احذف أي صف فيه قيمة ناقصة
print(df.isnull().sum().sum())

customer_id          0
age                  0
gender               0
tenure               0
usage_frequency      0
support_calls        0
payment_delay        0
subscription_type    0
contract_length      0
total_spend          0
last_interaction     0
churn                0
dtype: int64
0


In [37]:

# 5. data types (float -> int after removing the empty row)
int_cols = ["customer_id", "age", "tenure", "usage_frequency", "support_calls",
            "payment_delay", "total_spend", "last_interaction", "churn"]
df[int_cols] = df[int_cols].astype(int)

In [38]:
# 6. clean text columns
for col in ["gender", "subscription_type", "contract_length"]:
    df[col] = df[col].str.strip().str.title()

In [39]:
# 7. check impossible values
print(df[["age", "tenure", "support_calls", "payment_delay", "total_spend"]].min())

age               18
tenure             1
support_calls      0
payment_delay      0
total_spend      100
dtype: int64


In [40]:
# 8. new columns
df["churn_label"] = df["churn"].map({1: "Churned", 0: "Stayed"})
df["age_group"] = pd.cut(df["age"], [17, 25, 35, 45, 55, 65],
                         labels=["18-25", "26-35", "36-45", "46-55", "56-65"])
df["tenure_group"] = pd.cut(df["tenure"], [0, 12, 24, 36, 48, 60],
                            labels=["0-12m", "13-24m", "25-36m", "37-48m", "49-60m"])
df["spend_group"] = pd.cut(df["total_spend"], [0, 300, 600, 800, 1000],
                           labels=["Low", "Medium", "High", "Very High"])
df["support_calls_group"] = pd.cut(df["support_calls"], [-1, 2, 5, 10],
                                   labels=["Low (0-2)", "Medium (3-5)", "High (6-10)"])
df["payment_delay_group"] = pd.cut(df["payment_delay"], [-1, 10, 20, 30],
                                   labels=["Low (0-10d)", "Medium (11-20d)", "High (21-30d)"])
df["inactivity_group"] = pd.cut(df["last_interaction"], [0, 7, 14, 21, 30],
                                labels=["0-7d", "8-14d", "15-21d", "22-30d"])
df["avg_monthly_spend"] = (df["total_spend"] / df["tenure"]).round(2)


In [41]:
# 10. final checks
print(df.isnull().sum().sum(), "missing values left")
print(df.shape)


0 missing values left
(440832, 20)


In [42]:
# 11. save
df.to_csv("churn_clean.csv", index=False)
